In [9]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv
from tqdm import tqdm

In [10]:
# --- CONFIGURACIÓN ---
# Carga las variables de entorno (mi API_TOKEN en el archivo .env)
load_dotenv()
API_TOKEN = os.getenv("AQI_API_KEY")
BASE_URL = "https://api.waqi.info"

In [11]:
# Archivo de entrada (el que limpié manualmente)
archivo_entrada = 'lista_antenas.csv'
# Archivo de salida donde se guardarán los resultados
archivo_salida = 'antenas_con_coordenadas.csv'
# ---------------------


In [12]:
# Verificamos que el token de la API se haya cargado
if not API_TOKEN:
    print(" Error: No se encontró el API_TOKEN. Asegúrate de que tu archivo .env está en la misma carpeta y tiene la variable API_TOKEN.")
else:
    try:
        # Leemos el archivo con la lista de antenas limpias
        df_antenas = pd.read_csv(archivo_entrada)
        
        # Lista para ir guardando los resultados de cada antena
        resultados_finales = []

        print(f"Iniciando la búsqueda de coordenadas para {len(df_antenas)} antenas...")
        print("Esto puede tardar varios minutos...")

        # tqdm nos envuelve el bucle para mostrarnos una barra de progreso
        for nombre_antena in tqdm(df_antenas['antena']):
            lat, lon = None, None  # Inicializamos las coordenadas como nulas

            try:
                # Paso 1: Usar el endpoint de búsqueda de la API
                url_busqueda = f"{BASE_URL}/search/?token={API_TOKEN}&keyword={nombre_antena}"
                response_busqueda = requests.get(url_busqueda)
                response_busqueda.raise_for_status() # Lanza un error si la petición falla
                
                data_busqueda = response_busqueda.json()
                
                # Si la API encuentra resultados para esa búsqueda
                if data_busqueda.get("status") == "ok" and data_busqueda.get("data"):
                    # Tomamos el identificador del primer resultado (el más probable)
                    station_id = data_busqueda["data"][0]["station"]["url"]
                    
                    # Paso 2: Con el identificador, obtenemos los datos completos de la estación
                    url_feed = f"{BASE_URL}/feed/{station_id}/?token={API_TOKEN}"
                    response_feed = requests.get(url_feed)
                    response_feed.raise_for_status()
                    
                    data_feed = response_feed.json()
                    
                    if data_feed.get("status") == "ok":
                        # ¡Éxito! Extraemos la latitud y longitud
                        coords = data_feed["data"]["city"]["geo"]
                        lat, lon = coords[0], coords[1]

            except requests.exceptions.RequestException as e:
                # Si hay un error de red, lo imprimimos pero continuamos
                print(f"\nError de red buscando '{nombre_antena}': {e}")
            
            # Agregamos el resultado (con o sin coordenadas) a nuestra lista
            resultados_finales.append({
                'antena': nombre_antena,
                'latitud': lat,
                'longitud': lon
            })
            
            # Pausa de 0.5 segundos entre cada petición para no sobrecargar la API
            time.sleep(0.5)

        # Convertimos la lista de resultados en un DataFrame de Pandas
        df_final = pd.DataFrame(resultados_finales)

        # Guardamos el DataFrame en el nuevo archivo CSV
        df_final.to_csv(archivo_salida, index=False)

        print(f"\n Proceso completado, Se ha creado el archivo '{archivo_salida}'.")
        print("Algunas pueden estar vacías si la API no encontró la antena.")

    except FileNotFoundError:
        print(f" Error: No se encontró el archivo de entrada '{archivo_entrada}'. Asegúrate de que el nombre sea correcto.")


Iniciando la búsqueda de coordenadas para 86 antenas...
Esto puede tardar varios minutos...



100%|███████████████████████████████████████████| 86/86 [04:07<00:00,  2.88s/it]


 Proceso completado, Se ha creado el archivo 'antenas_con_coordenadas.csv'.
Algunas pueden estar vacías si la API no encontró la antena.
